# What is LangSmith?


> **LangSmith is a CCTV camera + flight recorder for your LLM app.**
> Every question, every document, every prompt, every reply - recorded, timed, and priced.

### How LangSmith maps to the 5 pillars of observability

| Pillar | In LangSmith |
|---|---|
| Logging | Every run stores its exact **inputs and outputs** |
| Tracing | The **trace tree** - its core feature |
| Metrics | Latency, tokens, **cost**, error rate |
| User feedback | `client.create_feedback(...)` attached to a run |
| Evaluation | **Datasets + evaluators** |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.environ.get("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_API_KEY"] = os.environ.get("LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "default")

for key_name in ["LANGSMITH_API_KEY"]:
    if os.environ.get(key_name):
        print(key_name, "loaded")
    else:
        print(key_name, "MISSING, add it to your .env file")

print("Tracing:", os.environ.get("LANGSMITH_TRACING"))
print("Project:", os.environ.get("LANGSMITH_PROJECT"))

OPENAI_API_KEY loaded
LANGSMITH_API_KEY loaded
Tracing: true
Project: observability


In [10]:
from langsmith import Client

client = Client()

print("connected")

connected


In [11]:
import time
from langsmith import traceable

@traceable(run_type="retriever")
def search_docs(question):
    time.sleep(0.2)                       
    return ["refund_policy.md", "faq.md"]

@traceable(run_type="llm")
def fake_llm(question, docs):
    time.sleep(0.4)                       
    return "Based on " + str(len(docs)) + " documents, here is the answer."

@traceable                                
def pipeline(question):
    docs = search_docs(question)          
    answer = fake_llm(question, docs)     
    return answer

print(pipeline("What is the refund policy?"))

Based on 2 documents, here is the answer.


build a RAG app

In [12]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0)

reply = llm.invoke("reply with exactly: langsmith is listening")
print(reply.content)

langsmith is listening


In [13]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

documents = [
    "Refunds are processed within 7 business days for orders placed under 30 days ago.",
    "Standard shipping takes 3 to 5 business days. Express shipping takes 1 business day.",
    "Our support team is available Monday to Friday, 9am to 6pm IST.",
    "You can cancel a subscription anytime from Settings. Billing stops at the end of the cycle.",
]

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

found = retriever.invoke("I want my money back, how long does it take?")

for doc in found:
    print("-", doc.page_content)

- Refunds are processed within 7 business days for orders placed under 30 days ago.
- Standard shipping takes 3 to 5 business days. Express shipping takes 1 business day.


In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support assistant. Answer ONLY using the context below.\n\n{context}"),
    ("human", "{question}"),
])

@traceable(name="rag_answer")
def rag_answer(question):

    docs = retriever.invoke(question)

    context = ""
    for doc in docs:
        context = context + "- " + doc.page_content + "\n"

    messages = prompt.format_messages(context=context, question=question)
    reply = llm.invoke(messages)

    return reply.content

In [15]:
print(rag_answer("How many days do refunds take?"))

Refunds are processed within 7 business days.


In [16]:
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of an order by its ID."""
    orders = {"A123": "Shipped, arriving Tuesday", "B456": "Processing"}
    if order_id in orders:
        return orders[order_id]
    return "Order not found"

@tool
def calculate_refund(price: float, days_since_purchase: int) -> str:
    """Calculate the refund amount for an order."""
    if days_since_purchase > 30:
        return "Not eligible: purchased more than 30 days ago."
    return "Eligible for a full refund of " + str(price)

tools = [get_order_status, calculate_refund]



In [17]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

llm_with_tools = llm.bind_tools(tools)       

def call_model(state):
    """The brain: look at the conversation so far and decide what to do."""
    reply = llm_with_tools.invoke(state["messages"])
    return {"messages": [reply]}


builder = StateGraph(MessagesState)

builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools))


builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

agent = builder.compile()

In [18]:
question = "My order A123 cost 2500 and I bought it 10 days ago. Where is it, and what refund would I get?"

final_state = agent.invoke({"messages": [("human", question)]})

for message in final_state["messages"]:
    message.pretty_print()

================================ Human Message =================================

My order A123 cost 2500 and I bought it 10 days ago. Where is it, and what refund would I get?
================================== Ai Message ==================================
Tool Calls:
  get_order_status (36b2a2d4-d5c0-4cfe-ac5d-fa2ed3dfbe37)
 Call ID: 36b2a2d4-d5c0-4cfe-ac5d-fa2ed3dfbe37
  Args:
    order_id: A123
  calculate_refund (43f8b85f-14fd-428e-814e-e0be8013e30b)
 Call ID: 43f8b85f-14fd-428e-814e-e0be8013e30b
  Args:
    price: 2500
    days_since_purchase: 10
================================= Tool Message =================================
Name: get_order_status

Shipped, arriving Tuesday
================================= Tool Message =================================
Name: calculate_refund

Eligible for a full refund of 2500.0
================================== Ai Message ==================================

The status of your order A123 is Shipped, and it is expected to arrive on Tuesday.

As